<a href="https://colab.research.google.com/github/nageshnnazare/cuda-bench/blob/main/colab/cuda_bench_matmul_launcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Nov 27 17:06:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!rm -rf /content/cuda-bench
%cd '/content'
!git clone https://github.com/nageshnnazare/cuda-bench.git '/content/cuda-bench/'
%cd '/content/cuda-bench'
!chmod +x scripts/profile.sh

/content
Cloning into '/content/cuda-bench'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 147 (delta 57), reused 99 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 567.70 KiB | 15.34 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/cuda-bench


In [3]:
!make matmul

mkdir -p bin
nvcc -O3 -std=c++17 -arch=sm_75 -Iinclude/ -lineinfo -Xptxas=-v,-warn-spills src/matmul/01_naive.cu -o bin/matmul_01_naive  2> bin/matmul_01_naive.ptxas.log || { cat bin/matmul_01_naive.ptxas.log; exit 1; }; grep -E "warning|spill|registers" bin/matmul_01_naive.ptxas.log | grep -v " 0 bytes spill" || true
ptxas info    : Used 52 registers, used 0 barriers, 392 bytes cmem[0]
nvcc -O3 -std=c++17 -arch=sm_75 -Iinclude/ -lineinfo -Xptxas=-v,-warn-spills src/matmul/02_coalesced.cu -o bin/matmul_02_coalesced  2> bin/matmul_02_coalesced.ptxas.log || { cat bin/matmul_02_coalesced.ptxas.log; exit 1; }; grep -E "warning|spill|registers" bin/matmul_02_coalesced.ptxas.log | grep -v " 0 bytes spill" || true
ptxas info    : Used 52 registers, used 0 barriers, 392 bytes cmem[0]
nvcc -O3 -std=c++17 -arch=sm_75 -Iinclude/ -lineinfo -Xptxas=-v,-warn-spills src/matmul/03_smem_tiling.cu -o bin/matmul_03_smem_tiling  2> bin/matmul_03_smem_tiling.ptxas.log || { cat bin/matmul_03_smem_tiling.ptx

In [4]:
!make run

mkdir -p reports
== bin/matmul_01_naive
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)
  [ ok ] v0 naive vs CPU              max_abs_err=1.431e-05

v0 naive                    278.039 ms    0.0618 TFLOP/s
== bin/matmul_02_coalesced
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)
  [ ok ] v1 coalesced vs CPU          max_abs_err=1.431e-05

v1 coalesced                 29.845 ms    0.5756 TFLOP/s
== bin/matmul_03_smem_tiling
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)
  [ ok ] v2 smem tiling vs CPU        max_abs_err=1.431e-05

v2 smem tiling               19.965 ms    0.8605 TFLOP/s


In [5]:
!make profile matmul

./scripts/profile.sh matmul_01_naive matmul_02_coalesced matmul_03_smem_tiling
== matmul_01_naive
### full report: ncu --set full --import-source yes
==PROF== Connected to process 921 (/content/cuda-bench/bin/matmul_01_naive)
==PROF== Profiling "sgemm_naive" - 0: 0%
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
....50%....100% - 33 passes
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)

v0 naive                  41810.113 ms    0.0004 TFLOP/s
==PROF== Disconnected from process 921
==PROF== Report: /content/cuda-bench/reports/matmul_Tesla-T4_clock-base/matmul_01_naive.ncu-rep
### metric table: ncu --metrics (the raw CSV is written to matmul_01_naive.csv)
| kernel | time (us) | SM % | Mem % | DRAM rd (MB) | DRAM wr (MB) | occupancy % | regs | s

In [6]:
%cd '/content/'
!rm -rf cuda-bench.zip
!zip -r 'cuda-bench.zip' 'cuda-bench/' > /dev/null

/content


In [7]:
from google.colab import files
files.download('/content/cuda-bench.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>